# LAB 02 — Language Models
N-gram language models, MLE, Laplace smoothing, sentence scores, perplexity, prediction, ranking, and corpus experiments.

Convention: no sentence-boundary tokens are added. Counts never cross sentence boundaries. Sentence scoring begins with a unigram, then uses the available preceding context. The training vocabulary is built from train documents only; validation/test OOV tokens map to <UNK>.


## 1. Imports

In [1]:
from pathlib import Path
import csv
import gzip
import json
import math
import random
import sys
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from ngram_lm import (
    NGramLanguageModel, UNK, build_vocabulary, count_ngrams,
    iter_tokenized_sentences, perplexity_from_log_probabilities,
    rank_candidates, train_bigram, train_trigram, train_unigram,
    unseen_ngram_statistics,
)
print("Imports ready; core model uses only the Python standard library.")


Imports ready; core model uses only the Python standard library.


## 2. Configuration

In [2]:
DATA_FILENAME = "c4-train.00000-of-01024-30K.json.gz"
SPLIT_SEED = 42
TRAIN_FRACTION = 0.80
VALIDATION_FRACTION = 0.10
TEST_FRACTION = 0.10
RESULTS_PATH = Path("results.csv")

cwd = Path.cwd()
data_candidates = [
    cwd.parent / "Lab01" / DATA_FILENAME,
    cwd / "Lab01" / DATA_FILENAME,
    cwd.parent.parent / "Lab01" / DATA_FILENAME,
]
DATA_PATH = next((path.resolve() for path in data_candidates if path.is_file()), None)
DATASET_AVAILABLE = DATA_PATH is not None
print({
    "dataset_path": str(DATA_PATH) if DATA_PATH else None,
    "split_seed": SPLIT_SEED,
    "split": {"train": TRAIN_FRACTION, "validation": VALIDATION_FRACTION, "test": TEST_FRACTION},
})


{'dataset_path': 'D:\\Study\\Semester 7\\NLP\\Lab01\\c4-train.00000-of-01024-30K.json.gz', 'split_seed': 42, 'split': {'train': 0.8, 'validation': 0.1, 'test': 0.1}}


## 3. Load dataset

In [3]:
def load_jsonl_gzip(path):
    records = []
    with gzip.open(path, "rt", encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            text = record.get("text")
            if not isinstance(text, str):
                raise ValueError(f"Record {line_number} has no string 'text' field")
            records.append(text)
    return records

if DATASET_AVAILABLE:
    documents = load_jsonl_gzip(DATA_PATH)
    print(f"Loaded {len(documents):,} documents from {DATA_PATH.name}")
else:
    documents = []
    print("DATASET_REQUIRED: place the Lab01 C4 JSONL.GZ corpus at the configured path.")


Loaded 30,000 documents from c4-train.00000-of-01024-30K.json.gz


## 4. Tokenization

In [4]:
if DATASET_AVAILABLE:
    tokenized_documents = [
        [[sys.intern(token) for token in sentence]
         for sentence in iter_tokenized_sentences(document)]
        for document in documents
    ]
    nonempty_sentence_count = sum(len(document) for document in tokenized_documents)
    token_total = sum(len(sentence) for document in tokenized_documents for sentence in document)
    print({
        "documents": len(documents), "sentences": nonempty_sentence_count, "tokens": token_total,
        "tokenizer": "NFKC + casefold + Unicode word tokens; punctuation separates words",
    })
else:
    tokenized_documents = []
    print("DATASET_REQUIRED")


{'documents': 30000, 'sentences': 635207, 'tokens': 10970394, 'tokenizer': 'NFKC + casefold + Unicode word tokens; punctuation separates words'}


## 5. Train / validation / test split

In [5]:
if DATASET_AVAILABLE:
    document_indices = list(range(len(tokenized_documents)))
    random.Random(SPLIT_SEED).shuffle(document_indices)
    train_end = int(len(document_indices) * TRAIN_FRACTION)
    validation_end = train_end + int(len(document_indices) * VALIDATION_FRACTION)
    train_document_indices = document_indices[:train_end]
    validation_document_indices = document_indices[train_end:validation_end]
    test_document_indices = document_indices[validation_end:]

    train_sentences = [sentence for index in train_document_indices for sentence in tokenized_documents[index]]
    validation_sentences = [sentence for index in validation_document_indices for sentence in tokenized_documents[index]]
    test_sentences = [sentence for index in test_document_indices for sentence in tokenized_documents[index]]
    data_splits = {"Train": train_sentences, "Validation": validation_sentences, "Test": test_sentences}
    print({
        "train_documents": len(train_document_indices),
        "validation_documents": len(validation_document_indices),
        "test_documents": len(test_document_indices),
        "train_sentences": len(train_sentences),
        "validation_sentences": len(validation_sentences),
        "test_sentences": len(test_sentences),
        "vocabulary_fit_source": "train only",
    })
else:
    train_sentences = validation_sentences = test_sentences = []
    data_splits = {}
    print("DATASET_REQUIRED")


{'train_documents': 24000, 'validation_documents': 3000, 'test_documents': 3000, 'train_sentences': 510891, 'validation_sentences': 62084, 'test_sentences': 62232, 'vocabulary_fit_source': 'train only'}


## 6. Corpus statistics

In [6]:
def collect_corpus_statistics(tokenized_docs):
    counts_by_order = {order: Counter() for order in (1, 2, 3)}
    sentence_count = 0
    token_count = 0
    for document in tokenized_docs:
        for sentence in document:
            sentence_count += 1
            token_count += len(sentence)
            counts_by_order[1].update(sentence)
            for order in (2, 3):
                counts_by_order[order].update(
                    tuple(sentence[index:index + order])
                    for index in range(len(sentence) - order + 1)
                )
    stats = {
        "number_of_documents": len(tokenized_docs),
        "number_of_sentences": sentence_count,
        "number_of_tokens": token_count,
        "vocabulary_size": len(counts_by_order[1]),
        "unique_unigrams": len(counts_by_order[1]),
        "unique_bigrams": len(counts_by_order[2]),
        "unique_trigrams": len(counts_by_order[3]),
        "unigram_types_once": sum(count == 1 for count in counts_by_order[1].values()),
        "bigram_types_once": sum(count == 1 for count in counts_by_order[2].values()),
        "trigram_types_once": sum(count == 1 for count in counts_by_order[3].values()),
    }
    frequency_histograms = {order: Counter(counts.values()) for order, counts in counts_by_order.items()}
    return stats, frequency_histograms

if DATASET_AVAILABLE:
    corpus_stats, frequency_histograms = collect_corpus_statistics(tokenized_documents)
    display(pd.DataFrame([{"statistic": key, "value": value} for key, value in corpus_stats.items()]))
    print("Corpus statistics describe all documents; they are descriptive only.")
else:
    corpus_stats, frequency_histograms = {}, {}
    print("DATASET_REQUIRED")


Corpus statistics describe all documents; they are descriptive only.


,statistic,value
0,number_of_documents,30000
1,number_of_sentences,635207
2,number_of_tokens,10970394
3,vocabulary_size,206107
4,unique_unigrams,206107
5,unique_bigrams,2816642
6,unique_trigrams,6492481
7,unigram_types_once,102248
8,bigram_types_once,2027065
9,trigram_types_once,5618041


## Analysis — Corpus and N-gram Statistics

### Câu hỏi 1 — Số lượng n-gram duy nhất thay đổi thế nào?

**Trả lời:** Số loại n-gram duy nhất tăng khi n tăng.

**Bằng chứng:**
- Unigram: 206,107; bigram: 2,816,642; trigram: 6,492,481.

**Diễn giải:** Chuỗi dài hơn tạo ra nhiều mẫu token và context riêng biệt hơn. Kích thước từ vựng token vẫn là 206,107.

### Câu hỏi 2 — Số singleton n-gram thay đổi thế nào?

**Trả lời:** N-gram dài hơn xuất hiện đúng một lần nhiều hơn.

**Bằng chứng:** Singleton unigram: 102,248; bigram: 2,027,065; trigram: 5,618,041.

**Diễn giải:** Các mẫu hiếm tăng mạnh theo n, rõ nhất ở trigram.

### Câu hỏi 3 — Độ thưa có tăng theo n không?

**Trả lời:** Có, số liệu cho thấy độ thưa tăng khi n tăng.

**Bằng chứng:** Số loại duy nhất tăng từ 206,107 lên 6,492,481; singleton tăng từ 102,248 lên 5,618,041. Số loại unseen trên validation là 1/196,437/562,440 và trên test là 1/196,621/560,193 theo thứ tự unigram/bigram/trigram.

**Diễn giải:** Context càng cụ thể thì càng khó lặp lại trong tập train.

## 7. Frequency distribution

In [7]:
if DATASET_AVAILABLE:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for axis, order, label in zip(axes, (1, 2, 3), ("Unigram", "Bigram", "Trigram")):
        histogram = frequency_histograms[order]
        frequencies = sorted(histogram)
        type_counts = [histogram[frequency] for frequency in frequencies]
        axis.loglog(frequencies, type_counts, marker=".", linestyle="none")
        axis.set_title(f"{label} frequency of frequencies")
        axis.set_xlabel("Occurrences per n-gram type")
        axis.set_ylabel("Number of n-gram types")
        axis.grid(True, which="both", alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print("DATASET_REQUIRED")


<ipython-input-1-382648f60f8c>:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Load and fit NGramLanguageModel

In [8]:
if DATASET_AVAILABLE:
    # Fit order 3 once, then use lightweight model views with shared counters.
    fitted_trigram = NGramLanguageModel(3, smoothing=None, unk_token=UNK).fit(train_sentences)

    def model_view(order, smoothing=None):
        view = NGramLanguageModel(order, smoothing=smoothing, unk_token=UNK)
        view.vocabulary = fitted_trigram.vocabulary
        view.ngram_counts = fitted_trigram.ngram_counts
        view.context_counts = fitted_trigram.context_counts
        view.token_count = fitted_trigram.token_count
        view.is_fitted = True
        return view

    print({
        "train_vocabulary_size_including_UNK": len(fitted_trigram.vocabulary),
        "train_tokens": fitted_trigram.token_count,
        "stored_ngram_types": {order: len(counts) for order, counts in fitted_trigram.ngram_counts.items()},
    })
else:
    fitted_trigram = None
    print("DATASET_REQUIRED")


{'train_vocabulary_size_including_UNK': 180197, 'train_tokens': 8853627, 'stored_ngram_types': {1: 180196, 2: 2388078, 3: 5360512}}


## 9. Verify hand-calculated examples

In [9]:
small_corpus = ["the cat eats fish", "the cat likes fish", "the dog eats meat"]
assert len(build_vocabulary(small_corpus)) == 7
small_unigram = train_unigram(small_corpus, unk_token=None)
small_bigram = train_bigram(small_corpus, unk_token=None)
small_trigram = train_trigram(small_corpus, unk_token=None)
assert math.isclose(small_unigram.probability([], "the"), 0.25)
assert math.isclose(small_bigram.probability(["the"], "cat"), 2 / 3)
assert math.isclose(small_bigram.probability(["the"], "dog"), 1 / 3)
assert math.isclose(small_bigram.probability(["cat"], "eats"), 0.5)
assert math.isclose(small_bigram.probability(["cat"], "likes"), 0.5)
assert math.isclose(small_bigram.probability(["eats"], "fish"), 0.5)
assert math.isclose(small_trigram.probability(["the", "cat"], "eats"), 0.5)
assert math.isclose(small_bigram.sentence_probability("the cat eats fish"), 1 / 24)
assert math.isclose(
    math.exp(small_bigram.sentence_log_probability("the cat eats fish")),
    small_bigram.sentence_probability("the cat eats fish"),
)
print("Hand-calculated vocabulary, n-gram, probability, sentence, and log checks passed.")


Hand-calculated vocabulary, n-gram, probability, sentence, and log checks passed.


## 10. Zero probability example

In [10]:
zero_corpus = ["I like NLP", "I like AI", "I study NLP"]
zero_model = train_bigram(zero_corpus, unk_token=None)
assert zero_model.ngram_counts[2][("study", "ai")] == 0
assert zero_model.probability(["study"], "AI") == 0.0
assert zero_model.sentence_probability("I study AI") == 0.0
assert zero_model.sentence_log_probability("I study AI") == -math.inf
print({
    "C(study, AI)": zero_model.ngram_counts[2][("study", "ai")],
    "P(AI | study)": zero_model.probability(["study"], "AI"),
    "sentence_log_probability": zero_model.sentence_log_probability("I study AI"),
})


{'C(study, AI)': 0, 'P(AI | study)': 0.0, 'sentence_log_probability': -inf}


## 11. Laplace smoothing example

In [11]:
laplace_zero_corpus = [["cat", "other"] for _ in range(10)] + [["eats"], ["a"], ["b"]]
laplace_zero = NGramLanguageModel(2, smoothing="laplace", unk_token=None).fit(laplace_zero_corpus)
laplace_three_corpus = [["cat", "eats"] for _ in range(3)]
laplace_three_corpus += [["cat", "other"] for _ in range(7)] + [["a"], ["b"]]
laplace_three = NGramLanguageModel(2, smoothing="laplace", unk_token=None).fit(laplace_three_corpus)
assert len(laplace_zero.vocabulary) == 5
assert laplace_zero.ngram_counts[1]["cat"] == 10
assert laplace_zero.ngram_counts[2][("cat", "eats")] == 0
assert math.isclose(laplace_zero.probability(["cat"], "eats"), 1 / 15)
assert math.isclose(laplace_three.probability(["cat"], "eats"), 4 / 15)
display(pd.DataFrame([
    {"C(cat, eats)": 0, "P_laplace": laplace_zero.probability(["cat"], "eats"), "expected": 1 / 15},
    {"C(cat, eats)": 3, "P_laplace": laplace_three.probability(["cat"], "eats"), "expected": 4 / 15},
]))


,"C(cat, eats)",P_laplace,expected
0,0,0.066667,0.066667
1,3,0.266667,0.266667


## 12. MLE vs Laplace experiment

In [12]:
experiment_rows = []
unseen_cache = {}

if DATASET_AVAILABLE:
    for order in (1, 2, 3):
        mle_view = model_view(order, smoothing=None)
        for split_name, split_corpus in data_splits.items():
            unseen_cache[(order, split_name)] = unseen_ngram_statistics(mle_view, split_corpus)

    for order in (2, 3):
        for smoothing, smoothing_label in ((None, "MLE"), ("laplace", "Laplace")):
            model = model_view(order, smoothing=smoothing)
            model_name = f"{'Bigram' if order == 2 else 'Trigram'} {smoothing_label}"
            for split_name, split_corpus in data_splits.items():
                unseen = unseen_cache[(order, split_name)]
                zero_events = unseen["zero_probability_events"] if smoothing is None else 0
                perplexity_value = model.perplexity(split_corpus)
                experiment_rows.append({
                    "experiment": "mle_vs_laplace", "model": model_name, "n": order,
                    "smoothing": smoothing_label, "split": split_name, "metric": "perplexity",
                    "value": perplexity_value, "unseen_ngram_types": unseen["unseen_ngram_types"],
                    "unseen_ngram_events": unseen["unseen_ngram_events"],
                    "zero_probability_events": zero_events,
                })
                for metric, value in (
                    ("unseen_ngram_types", unseen["unseen_ngram_types"]),
                    ("unseen_ngram_events", unseen["unseen_ngram_events"]),
                    ("zero_probability_events", zero_events),
                ):
                    experiment_rows.append({
                        "experiment": "mle_vs_laplace", "model": model_name, "n": order,
                        "smoothing": smoothing_label, "split": split_name, "metric": metric,
                        "value": value,
                    })

    wide_rows = []
    for row in experiment_rows:
        if row["metric"] == "perplexity":
            wide_rows.append({"Model": row["model"], f"{row['split']} PPL": row["value"]})
    mle_laplace_table = pd.DataFrame(wide_rows).groupby("Model", as_index=False).first()
    display(mle_laplace_table)
    display(pd.DataFrame([
        {
            "Model": row["model"], "Split": row["split"],
            "Unseen n-gram types": row["value"],
            "Unseen n-gram events": row["unseen_ngram_events"],
            "Zero-probability events": row["zero_probability_events"],
        }
        for row in experiment_rows if row["metric"] == "perplexity"
    ]))
else:
    print("DATASET_REQUIRED")


,Model,Train PPL,Validation PPL,Test PPL
0,Bigram Laplace,6505.709546,8.567008e+03,8.578537e+03
1,Bigram MLE,192.215446,inf,inf
2,Trigram Laplace,32469.955602,5.390084e+04,5.406413e+04
3,Trigram MLE,16.455391,inf,inf


,Model,Split,Unseen n-gram types,Unseen n-gram events,Zero-probability events
0,Bigram MLE,Train,1.922154e+02,0,0
1,Bigram MLE,Validation,inf,246446,248700
2,Bigram MLE,Test,inf,246990,249501
3,Bigram Laplace,Train,6.505710e+03,0,0
4,Bigram Laplace,Validation,8.567008e+03,246446,0
5,Bigram Laplace,Test,8.578537e+03,246990,0
6,Trigram MLE,Train,1.645539e+01,0,0
7,Trigram MLE,Validation,inf,604191,620616
8,Trigram MLE,Test,inf,600977,617974
9,Trigram Laplace,Train,3.246996e+04,0,0


## Analysis — MLE vs Laplace Smoothing

### Câu hỏi 1 — Trên tập train

**Trả lời:** Laplace có perplexity train cao hơn nhiều ở cả hai bậc.

**Bằng chứng:** Bigram: MLE 192.215446, Laplace 6,505.709546. Trigram: MLE 16.455391, Laplace 32,469.955602.

**Diễn giải:** Add-one phân bổ xác suất cho cả các kết quả chưa gặp trên từ vựng train lớn gồm 180,197 mục (tính cả `<UNK>`), làm giảm xác suất dành cho các sự kiện đã quan sát.

### Câu hỏi 2 — Trên validation và test

**Trả lời:** PPL của MLE là `inf`; PPL của Laplace hữu hạn nhưng lớn.

**Bằng chứng:** Bigram validation/test: MLE `inf`/`inf`, Laplace 8,567.008155/8,578.536676. Trigram: MLE `inf`/`inf`, Laplace 53,900.835588/54,064.126875.

**Diễn giải:** Trong thí nghiệm này, Laplace tránh PPL vô hạn trên held-out, nhưng các giá trị lớn không cho thấy khả năng dự đoán tốt.

### Câu hỏi 3 — Smoothing có cải thiện PPL held-out không?

**Trả lời:** Mẫu số liệu thực tế là MLE có PPL train thấp hơn, còn Laplace cho PPL validation/test hữu hạn trong khi MLE là `inf`. Vì PPL MLE held-out không hữu hạn nên không thể so sánh định lượng chất lượng dự đoán giữa hai phương pháp.

### Câu hỏi 4 — Smoothing xử lý zero-frequency ra sao?

**Trả lời:** Output xác nhận Laplace loại bỏ các sự kiện xác suất bằng 0 trong phép chấm held-out này.

**Bằng chứng:** Bigram MLE có 246,446/246,990 unseen n-gram events và 248,700/249,501 zero-probability events trên validation/test. Trigram MLE lần lượt có 604,191/600,977 unseen events và 620,616/617,974 zero-probability events. Laplace ghi nhận 0 zero-probability events và PPL hữu hạn.

**Diễn giải:** N-gram unseen nhận xác suất MLE bằng 0; câu chứa sự kiện đó có xác suất tích bằng 0 và làm PPL MLE thành `inf`. Laplace gán xác suất dương, phù hợp với PPL hữu hạn trong output.

## 13. Perplexity experiment

In [13]:
known_ppl_1 = perplexity_from_log_probabilities([math.log(0.5), math.log(0.25), math.log(0.5)])
known_ppl_2 = perplexity_from_log_probabilities([math.log(0.5), math.log(0.1), math.log(0.5)])
assert math.isclose(known_ppl_1, 2.5198421, rel_tol=1e-6)
assert math.isclose(known_ppl_2, 3.4199519, rel_tol=1e-6)
print({"PPL case 1": known_ppl_1, "PPL case 2": known_ppl_2})

if DATASET_AVAILABLE:
    display(pd.DataFrame([
        {"Model": row["model"], "Split": row["split"], "Perplexity": row["value"]}
        for row in experiment_rows if row["metric"] == "perplexity"
    ]))
else:
    print("DATASET_REQUIRED")


{'PPL case 1': 2.519842099789746, 'PPL case 2': 3.4199518933533932}


,Model,Split,Perplexity
0,Bigram MLE,Train,1.922154e+02
1,Bigram MLE,Validation,inf
2,Bigram MLE,Test,inf
3,Bigram Laplace,Train,6.505710e+03
4,Bigram Laplace,Validation,8.567008e+03
5,Bigram Laplace,Test,8.578537e+03
6,Trigram MLE,Train,1.645539e+01
7,Trigram MLE,Validation,inf
8,Trigram MLE,Test,inf
9,Trigram Laplace,Train,3.246996e+04


## 14. Unigram vs Bigram vs Trigram

In [14]:
ngram_comparison_rows = []

if DATASET_AVAILABLE:
    for order, name in ((1, "Unigram"), (2, "Bigram"), (3, "Trigram")):
        model = model_view(order, smoothing=None)
        unique_count = len(fitted_trigram.ngram_counts[order])
        for split_name, split_corpus in data_splits.items():
            unseen = unseen_cache[(order, split_name)]
            ngram_comparison_rows.append({
                "Model": name, "n": order, "Split": split_name,
                "Perplexity": model.perplexity(split_corpus),
                "Unique training n-grams": unique_count,
                "Unseen n-gram types": unseen["unseen_ngram_types"],
                "Unseen n-gram events": unseen["unseen_ngram_events"],
                "Zero-probability events": unseen["zero_probability_events"],
            })
    display(pd.DataFrame(ngram_comparison_rows))
else:
    print("DATASET_REQUIRED")


,Model,n,Split,Perplexity,Unique training n-grams,Unseen n-gram types,Unseen n-gram events,Zero-probability events
0,Unigram,1,Train,2154.883513,180196,0,0,0
1,Unigram,1,Validation,inf,180196,1,21212,21212
2,Unigram,1,Test,inf,180196,1,21972,21972
3,Bigram,2,Train,192.215446,2388078,0,0,0
4,Bigram,2,Validation,inf,2388078,196437,246446,248700
5,Bigram,2,Test,inf,2388078,196621,246990,249501
6,Trigram,3,Train,16.455391,5360512,0,0,0
7,Trigram,3,Validation,inf,5360512,562440,604191,620616
8,Trigram,3,Test,inf,5360512,560193,600977,617974


## Analysis — Unigram vs Bigram vs Trigram

### Câu hỏi 1 — PPL train

**Trả lời:** PPL train giảm khi n tăng.

**Bằng chứng:** Unigram 2,154.883513; bigram 192.215446; trigram 16.455391.

**Diễn giải:** Model bậc cao khớp dữ liệu train sát hơn.

### Câu hỏi 2 — PPL validation

**Trả lời:** Validation không cải thiện tương tự; cả ba giá trị đều là `inf`: unigram `inf`, bigram `inf`, trigram `inf`.

### Câu hỏi 3 — PPL test

**Trả lời:** Không đo được cải thiện hữu hạn: PPL test của unigram, bigram và trigram đều là `inf`.

### Câu hỏi 4 — Có dấu hiệu overfitting không?

**Trả lời:** Có mẫu kết quả phù hợp với overfitting hoặc sparsity: PPL train giảm mạnh trong khi PPL validation/test vẫn vô hạn. Tuy nhiên, zero-probability events chi phối PPL held-out, nên các số liệu này chưa xếp hạng được khả năng tổng quát hóa giữa các bậc một cách rõ ràng.

## 15. Unseen n-gram statistics

In [15]:
if DATASET_AVAILABLE:
    display(pd.DataFrame([
        {
            "Model": name, "Split": split_name,
            "Unique n-grams in train": len(fitted_trigram.ngram_counts[order]),
            "Unseen n-gram types": unseen_cache[(order, split_name)]["unseen_ngram_types"],
            "Unseen n-gram events": unseen_cache[(order, split_name)]["unseen_ngram_events"],
            "Zero-probability events": unseen_cache[(order, split_name)]["zero_probability_events"],
        }
        for order, name in ((1, "Unigram"), (2, "Bigram"), (3, "Trigram"))
        for split_name in ("Validation", "Test")
    ]))
else:
    print("DATASET_REQUIRED")


,Model,Split,Unique n-grams in train,Unseen n-gram types,Unseen n-gram events,Zero-probability events
0,Unigram,Validation,180196,1,21212,21212
1,Unigram,Test,180196,1,21972,21972
2,Bigram,Validation,2388078,196437,246446,248700
3,Bigram,Test,2388078,196621,246990,249501
4,Trigram,Validation,5360512,562440,604191,620616
5,Trigram,Test,5360512,560193,600977,617974


## Analysis — Unseen N-grams

### Câu hỏi 1 — Validation

**Trả lời:** Trigram có nhiều loại unseen nhất.

**Bằng chứng:** Unigram 1; bigram 196,437; trigram 562,440 unseen types.

### Câu hỏi 2 — Test

**Trả lời:** Trigram cũng có nhiều loại unseen nhất trên test.

**Bằng chứng:** Unigram 1; bigram 196,621; trigram 560,193 unseen types.

### Câu hỏi 3 — Unseen n-gram liên hệ thế nào với perplexity?

**Trả lời:** MLE bậc cao có nhiều unseen types/events hơn và cả ba bậc đều có PPL held-out `inf`. Vì cùng là `inf`, PPL MLE không phân biệt được mức độ giữa các bậc. Với Laplace, PPL test của bigram là 8,578.536676 và trigram là 54,064.126875; kết quả này cùng chiều với sparsity cao hơn ở trigram trong thí nghiệm này.

## Analysis — Does Longer Context Always Help?

### Bằng chứng

- **Unigram:** PPL train 2,154.883513; validation/test `inf`/`inf`; 180,196 n-gram train duy nhất; 102,248 singleton trên toàn corpus; unseen types validation/test 1/1 (unseen events 21,212/21,972).
- **Bigram:** PPL train 192.215446; validation/test `inf`/`inf`; 2,388,078 n-gram train duy nhất; 2,027,065 singleton trên toàn corpus; unseen types validation/test 196,437/196,621 (unseen events 246,446/246,990).
- **Trigram:** PPL train 16.455391; validation/test `inf`/`inf`; 5,360,512 n-gram train duy nhất; 5,618,041 singleton trên toàn corpus; unseen types validation/test 562,440/560,193 (unseen events 604,191/600,977).

### Diễn giải

1. Context dài hơn giúp khớp train tốt hơn: PPL giảm từ 2,154.883513 xuống 16.455391.
2. Chưa đo được cải thiện trên validation/test: PPL MLE của cả ba bậc đều `inf`.
3. Sparsity tăng: số n-gram train duy nhất và singleton toàn corpus đều lớn hơn nhiều ở bậc cao.
4. Số loại unseen trên validation/test cũng tăng từ unigram lên trigram.
5. Đây là dấu hiệu cần thận trọng về khả năng tổng quát hóa, nhưng PPL vô hạn không cho phép so sánh hữu hạn giữa từng bậc.

### Kết luận

**Trả lời:** Context dài hơn không phải lúc nào cũng tốt hơn. Trong thí nghiệm này, nó làm giảm PPL train nhưng PPL MLE held-out vẫn `inf` và số mẫu unseen tăng.

## 16. Next-word prediction

In [16]:
prediction_rows = []

def find_actual_next(context_tokens, sentences):
    width = len(context_tokens)
    for sentence in sentences:
        for start in range(len(sentence) - width):
            if sentence[start:start + width] == context_tokens:
                return sentence[start + width]
    return None

if DATASET_AVAILABLE:
    required_contexts = ["the cat", "natural language", "machine learning"]
    dataset_contexts = []
    for gram, _count in fitted_trigram.ngram_counts[2].most_common():
        if UNK in gram:
            continue
        context = " ".join(gram)
        if context not in required_contexts and context not in dataset_contexts:
            dataset_contexts.append(context)
        if len(dataset_contexts) == 2:
            break
    prediction_contexts = required_contexts + dataset_contexts
    trigram_laplace = model_view(3, smoothing="laplace")
    for context in prediction_contexts:
        actual = find_actual_next(context.split(), test_sentences)
        for rank, (prediction, probability_value) in enumerate(
            trigram_laplace.predict_next(context, top_k=5), start=1
        ):
            prediction_rows.append({
                "experiment": "next_word_prediction", "model": "Trigram Laplace",
                "n": 3, "smoothing": "Laplace", "context": context, "rank": rank,
                "prediction": prediction, "actual": actual or "", "probability": probability_value,
            })
    prediction_diagnostic_rows = []
    top_predictions = {
        row["context"]: row for row in prediction_rows if row["rank"] == 1
    }
    for context in prediction_contexts:
        actual = next(row["actual"] for row in prediction_rows if row["context"] == context)
        top = top_predictions[context]
        mapped_context = []
        for token in trigram_laplace._input_tokens(context):
            mapped = trigram_laplace._map_token(token)
            if mapped is not None:
                mapped_context.append(mapped)
        history_key = tuple(mapped_context[-2:])
        context_count = trigram_laplace.context_counts[3].get(history_key, 0)
        predicted_token = trigram_laplace._map_token(top["prediction"])
        actual_token = trigram_laplace._map_token(actual) if actual else None
        predicted_count = trigram_laplace.ngram_counts[3].get(
            history_key + (predicted_token,), 0
        ) if predicted_token is not None else 0
        actual_count = trigram_laplace.ngram_counts[3].get(
            history_key + (actual_token,), 0
        ) if actual_token is not None else 0
        prediction_diagnostic_rows.append({
            "context": context,
            "train_context_count": context_count,
            "top1_prediction": top["prediction"],
            "top1_probability": top["probability"],
            "top1_trigram_count": predicted_count,
            "actual_next_word": actual or "",
            "actual_in_train_vocabulary": bool(actual and actual in trigram_laplace.vocabulary),
            "actual_trigram_count": actual_count,
            "actual_probability": trigram_laplace.probability(context, actual) if actual else None,
        })
    display(pd.DataFrame(prediction_rows))
    display(pd.DataFrame(prediction_diagnostic_rows))
    assert len(prediction_rows) >= 25

    valid_trigram_context = next(iter(fitted_trigram.context_counts[3]))
    valid_context_text = " ".join(valid_trigram_context)
    for smoothing in (None, "laplace"):
        distribution = model_view(3, smoothing=smoothing).next_word_distribution(valid_context_text)
        assert math.isclose(sum(distribution.values()), 1.0, abs_tol=1e-12)
    print(f"MLE and Laplace distributions sum to 1 for valid context: {valid_context_text}")
else:
    print("DATASET_REQUIRED")


MLE and Laplace distributions sum to 1 for valid context: after much


,experiment,model,n,smoothing,context,rank,prediction,actual,probability
0,next_word_prediction,Trigram Laplace,3,Laplace,the cat,1,box,family,0.000028
1,next_word_prediction,Trigram Laplace,3,Laplace,the cat,2,and,family,0.000022
2,next_word_prediction,Trigram Laplace,3,Laplace,the cat,3,can,family,0.000022
3,next_word_prediction,Trigram Laplace,3,Laplace,the cat,4,in,family,0.000022
4,next_word_prediction,Trigram Laplace,3,Laplace,the cat,5,is,family,0.000022
5,next_word_prediction,Trigram Laplace,3,Laplace,natural language,1,processing,analysis,0.000033
6,next_word_prediction,Trigram Laplace,3,Laplace,natural language,2,laboratory,analysis,0.000022
7,next_word_prediction,Trigram Laplace,3,Laplace,natural language,3,1980,analysis,0.000011
8,next_word_prediction,Trigram Laplace,3,Laplace,natural language,4,apps,analysis,0.000011
9,next_word_prediction,Trigram Laplace,3,Laplace,natural language,5,speech,analysis,0.000011


,context,train_context_count,top1_prediction,top1_probability,top1_trigram_count,actual_next_word,actual_in_train_vocabulary,actual_trigram_count,actual_probability
0,the cat,70,box,0.000028,4,family,True,0,0.000006
1,natural language,12,processing,0.000033,5,analysis,True,0,0.000006
2,machine learning,75,and,0.000061,10,portfolio,True,0,0.000006
3,of the,49559,most,0.004222,969,valentino,True,0,0.000004
4,in the,37139,world,0.003575,776,neighbourhood,True,5,0.000028


## Analysis — Next-word Prediction

### Câu hỏi 1 — Độ chính xác Top-1

**Trả lời:** Số dự đoán Top-1 đúng: 0/5.

### Câu hỏi 2 — Context được dự đoán đúng

**Trả lời:** Không có context nào được dự đoán đúng Top-1 trong 5 trường hợp.

**Bằng chứng:**
- `the cat`: thực tế `family`, dự đoán `box` (p=0.00002774).
- `natural language`: thực tế `analysis`, dự đoán `processing` (p=0.00003329).
- `machine learning`: thực tế `portfolio`, dự đoán `and` (p=0.00006102).
- `of the`: thực tế `valentino`, dự đoán `most` (p=0.00422187).
- `in the`: thực tế `neighbourhood`, dự đoán `world` (p=0.00357511).

### Câu hỏi 3 — Context được dự đoán sai

**Trả lời:** Cả 5 đều sai; ví dụ:

- `the cat`: thực tế `family`, dự đoán `box`.
- `natural language`: thực tế `analysis`, dự đoán `processing`.

### Câu hỏi 4 — Nguyên nhân có thể

**Trả lời:** Số liệu ủng hộ nguyên nhân là transition thực tế chưa gặp hoặc ít gặp trong train, không phải từ thực tế vắng mặt khỏi vocabulary train.

**Bằng chứng:**
- `the cat` (context train count 70): `family` count 0; `box` count 4; từ thực tế có trong vocabulary train.
- `natural language` (12): `analysis` count 0; `processing` count 5; có trong vocabulary.
- `machine learning` (75): `portfolio` count 0; `and` count 10; có trong vocabulary.
- `of the` (49,559): `valentino` count 0; `most` count 969; có trong vocabulary.
- `in the` (37,139): `neighbourhood` count 5; `world` count 776; có trong vocabulary.

**Diễn giải:** 4/5 transition thực tế chưa xuất hiện trong train; trường hợp còn lại có count 5 so với 776 của dự đoán Top-1. Context `natural language` có count thấp (12), nhưng các context rất phổ biến như `of the` và `in the` cũng bị dự đoán sai. Laplace cho xác suất dương cho từ thực tế nhưng không đảm bảo từ đó đứng Top-1. Số liệu không ủng hộ nguyên nhân OOV; sparsity và chênh lệch tần suất có bằng chứng trực tiếp.

## 17. Sentence ranking

In [17]:
ranking_rows = []
if DATASET_AVAILABLE:
    candidate_context = "machine learning"
    candidates = ["is useful for NLP", "banana computer quickly", "studies language models"]
    ranked_candidates = rank_candidates(
        candidate_context, candidates, model_view(3, smoothing="laplace")
    )
    for result in ranked_candidates:
        ranking_rows.append({
            "experiment": "sentence_ranking", "model": "Trigram Laplace",
            "n": 3, "smoothing": "Laplace", "context": candidate_context,
            "rank": result["rank"], "prediction": result["candidate"],
            "raw_log_probability": result["raw_log_probability"],
            "average_log_probability": result["average_log_probability"],
        })
    display(pd.DataFrame(ranking_rows))

    ranking_model = model_view(3, smoothing="laplace")
    candidate_event_rows = []
    for candidate in candidates:
        history = list(ranking_model._input_tokens(candidate_context))
        for position, token in enumerate(ranking_model._input_tokens(candidate), start=1):
            mapped_history = []
            for history_token in history[-2:]:
                mapped = ranking_model._map_token(history_token)
                if mapped is not None:
                    mapped_history.append(mapped)
            mapped_target = ranking_model._map_token(token)
            context_key = tuple(mapped_history)
            ngram_count = ranking_model.ngram_counts[3].get(
                context_key + (mapped_target,), 0
            ) if mapped_target is not None else 0
            candidate_event_rows.append({
                "candidate": candidate,
                "position": position,
                "context_in_train_vocab": " ".join(context_key),
                "target": token,
                "mapped_target": mapped_target,
                "train_context_count": ranking_model.context_counts[3].get(context_key, 0),
                "observed_trigram_count": ngram_count,
                "laplace_probability": ranking_model.probability(history, token),
                "trigram_unseen": ngram_count == 0,
            })
            history.append(token)
    display(pd.DataFrame(candidate_event_rows))
else:
    print("DATASET_REQUIRED")


,experiment,model,n,smoothing,context,rank,prediction,raw_log_probability,average_log_probability
0,sentence_ranking,Trigram Laplace,3,Laplace,machine learning,1,is useful for NLP,-45.464150,-11.366037
1,sentence_ranking,Trigram Laplace,3,Laplace,machine learning,2,banana computer quickly,-36.305834,-12.101945
2,sentence_ranking,Trigram Laplace,3,Laplace,machine learning,3,studies language models,-36.305840,-12.101947


,candidate,position,context_in_train_vocab,target,mapped_target,train_context_count,observed_trigram_count,laplace_probability,trigram_unseen
0,is useful for NLP,1,machine learning,is,is,75,0,0.000006,True
1,is useful for NLP,2,learning is,useful,useful,22,0,0.000006,True
2,is useful for NLP,3,is useful,for,for,59,18,0.000105,False
3,is useful for NLP,4,useful for,nlp,nlp,90,0,0.000006,True
4,banana computer quickly,1,machine learning,banana,banana,75,0,0.000006,True
5,banana computer quickly,2,learning banana,computer,computer,0,0,0.000006,True
6,banana computer quickly,3,banana computer,quickly,quickly,0,0,0.000006,True
7,studies language models,1,machine learning,studies,studies,75,0,0.000006,True
8,studies language models,2,learning studies,language,language,0,0,0.000006,True
9,studies language models,3,studies language,models,models,1,0,0.000006,True


## Analysis — Sentence Ranking

**Context:** `machine learning` (Trigram Laplace)

- **Candidate A — `is useful for NLP`:** raw log probability -45.464150; average -11.366037.
- **Candidate B — `banana computer quickly`:** raw -36.305834; average -12.101945.
- **Candidate C — `studies language models`:** raw -36.305840; average -12.101947.

### Câu hỏi 1 — Average score cao nhất

**Trả lời:** Candidate A có average log probability cao nhất.

### Câu hỏi 2 — Ranking có hợp trực giác không?

**Trả lời:** Candidate A có một transition trigram đã quan sát được; B và C không có transition trigram nào đã quan sát trong train.

**Bằng chứng:**
- Candidate A: 1/4 trigram đã quan sát; `is useful -> for` có count 18 và Laplace p=0.000105; 3/4 transition còn lại unseen.
- Candidate B: 0/3 đã quan sát; cả 3 transition unseen, mỗi transition có Laplace p xấp xỉ 0.000006.
- Candidate C: 0/3 đã quan sát; cả 3 transition unseen, mỗi transition có Laplace p xấp xỉ 0.000006.

**Diễn giải:** Transition `is useful -> for` đã xuất hiện trong train và có xác suất Laplace cao hơn; các transition của B/C đều chưa gặp. Bằng chứng này giải thích vì sao A có average score cao hơn trong thí nghiệm, nhưng không khẳng định chất lượng ngôn ngữ tổng quát.

### Câu hỏi 3 — Raw và average có cho thứ hạng khác nhau không?

**Trả lời:** Có. Raw xếp B, C, A; average xếp A, B, C. Candidate A có 4 token, còn B và C có 3. Raw cộng log probability của mọi token nên câu dài thường có tổng điểm âm hơn; average chia tổng điểm cho số token.

## 18. Save results.csv

In [18]:
CSV_COLUMNS = [
    "experiment", "model", "n", "smoothing", "split", "metric", "value",
    "context", "prediction", "actual", "rank", "probability",
    "raw_log_probability", "average_log_probability",
    "unseen_ngram_types", "unseen_ngram_events", "zero_probability_events",
]
csv_rows = []

if DATASET_AVAILABLE:
    for metric, value in corpus_stats.items():
        csv_rows.append({
            "experiment": "corpus_statistics", "model": "All documents",
            "split": "All", "metric": metric, "value": value,
        })
    for order, name in ((1, "Unigram"), (2, "Bigram"), (3, "Trigram")):
        csv_rows.append({
            "experiment": "unique_ngram_counts", "model": name, "n": order,
            "smoothing": "MLE", "split": "Train",
            "metric": "unique_training_ngrams",
            "value": len(fitted_trigram.ngram_counts[order]),
        })
    csv_rows.extend(experiment_rows)
    for comparison in ngram_comparison_rows:
        for metric_name, metric_value in (
            ("perplexity", comparison["Perplexity"]),
            ("unseen_ngram_types", comparison["Unseen n-gram types"]),
            ("unseen_ngram_events", comparison["Unseen n-gram events"]),
            ("zero_probability_events", comparison["Zero-probability events"]),
        ):
            csv_rows.append({
                "experiment": "unigram_vs_bigram_vs_trigram",
                "model": comparison["Model"],
                "n": comparison["n"],
                "smoothing": "MLE",
                "split": comparison["Split"],
                "metric": metric_name,
                "value": metric_value,
                "unseen_ngram_types": comparison["Unseen n-gram types"],
                "unseen_ngram_events": comparison["Unseen n-gram events"],
                "zero_probability_events": comparison["Zero-probability events"],
            })
    csv_rows.extend(prediction_rows)
    csv_rows.extend(ranking_rows)
else:
    csv_rows.append({
        "experiment": "dataset_status", "metric": "status", "value": "DATASET_REQUIRED",
    })

with RESULTS_PATH.open("w", newline="", encoding="utf-8") as output:
    writer = csv.DictWriter(output, fieldnames=CSV_COLUMNS)
    writer.writeheader()
    writer.writerows(csv_rows)
print(f"Saved {len(csv_rows):,} rows to {RESULTS_PATH.resolve()}")
print("AI assistance statement")
print("Tool: Codex (OpenAI coding agent)")
print("Purpose: Implementation support, debugging, testing and experiment setup.")
print("Generated: ngram_lm.py, experiments.ipynb, results.csv, test_ngram_lm.py.")
print("Modified: the four files listed above in Lab02.")
print("Verification: hand-calculated checks, unit tests, and notebook execution.")


Saved 125 rows to D:\Study\Semester 7\NLP\Lab02\results.csv
AI assistance statement
Tool: Codex (OpenAI coding agent)
Purpose: Implementation support, debugging, testing and experiment setup.
Generated: ngram_lm.py, experiments.ipynb, results.csv, test_ngram_lm.py.
Modified: the four files listed above in Lab02.
Verification: hand-calculated checks, unit tests, and notebook execution.


## Experimental Findings vs Initial Predictions

### 1. Vocabulary có tăng từ unigram → bigram → trigram không?

**Kết quả quan sát:** Kích thước token vocabulary giữ ở 206,107; unique n-gram types tăng từ 206,107 lên 2,816,642 rồi 6,492,481.

**Được thí nghiệm ủng hộ:** Chưa thể đánh giá khi thiếu dự đoán ban đầu.

**Bằng chứng:** Vocabulary token không tăng theo order; số pattern n-gram tăng.

### 2. Unique n-gram thay đổi thế nào?

**Kết quả quan sát:** Unigram 206,107; bigram 2,816,642; trigram 6,492,481.

**Được thí nghiệm ủng hộ:** Chưa thể đánh giá khi thiếu dự đoán ban đầu.

**Bằng chứng:** N-gram dài hơn tạo ra nhiều kiểu chuỗi khác nhau.

### 3. Model nào gặp zero probability nhiều hơn?

**Kết quả quan sát:** Validation: unigram 21,212; bigram 248,700; trigram 620,616. Test: lần lượt 21,972; 249,501; 617,974.

**Được thí nghiệm ủng hộ:** Chưa thể đánh giá khi thiếu dự đoán ban đầu.

**Bằng chứng:** Trigram có nhiều zero-probability events nhất trên cả validation và test.

### 4. Model nào có training perplexity thấp hơn?

**Kết quả quan sát:** Unigram 2,154.883513; bigram 192.215446; trigram 16.455391.

**Được thí nghiệm ủng hộ:** Chưa thể đánh giá khi thiếu dự đoán ban đầu.

**Bằng chứng:** Trigram có PPL train thấp nhất trong lần chạy này.

### 5. Trigram có chắc chắn tốt hơn bigram trên corpus nhỏ không?

**Kết quả quan sát:** PPL MLE validation/test của bigram và trigram đều `inf`. Với Laplace, PPL test bigram là 8,578.536676, trigram là 54,064.126875.

**Được thí nghiệm ủng hộ:** Chưa thể đánh giá dự đoán ban đầu.

**Bằng chứng:** Không thể khẳng định trigram tốt hơn: PPL train của trigram thấp hơn, nhưng PPL Laplace test cao hơn bigram trong lần chạy này.